In [1]:
import os
import re
import json
import math
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# PDF ingestion
from pypdf import PdfReader

# Embeddings
from sklearn.feature_extraction.text import TfidfVectorizer
#from sentence_transformers import SentenceTransformer

# (Optional) local generator
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# Distributed computing system. 
from pyspark.sql import SparkSession 
from pyspark import SparkContext, SparkConf
from pathlib import Path


In [2]:
@dataclass
class RAGConfig:
    pdf_path: str = "course_material"

    # Chunking (simple word-based chunking for this practical)
    chunk_size_words: int = 200
    chunk_overlap_words: int = 50

    # Retrieval
    top_k: int = 4
    similarity_metric: str = "cosine"   # {"cosine", "euclidean"}
    embedding_strategy: str = "sbert"   # {"tfidf", "sbert"}

    # Embedding models
    sbert_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"

    # Generator (fallback: small, CPU-friendly)
    gen_model_name: str = "google/flan-t5-small"
    max_new_tokens: int = 256

cfg = RAGConfig()

In [3]:
# Init Park
if SparkContext._active_spark_context:
    SparkContext._active_spark_context.stop()
    print("SparkContext existant arrêté.")
    
conf = SparkConf() \
    .setAppName("IngestData") \
    .setMaster("local[*]")

sc = SparkContext(conf=conf)
print("Spark version:", sc.version)

Spark version: 3.3.2


# Quelques fonctions
- sc.parallelize(c, numSlices=None)
    Elle prend deux paramètres :
    - 1. c — la collection (obligatoire)
    N'importe quelle collection Python itérable : list, range, tuple, string, dict, etc

    - 2. numSlices — le nombre de partitions (optionnel)
    Contrôle en combien de morceaux les données sont découpées. Si non spécifié, Spark décide automatiquement selon le nombre de cœurs disponibles.

    - Ce qu'elle retourne
    Un RDD — sur lequel vous pouvez ensuite chaîner des transformations (map, filter, flatMap...) ou des actions (collect, reduce, count...).

The pipeline must keep traceability metadata for every extracted unit, including at least the source PDF
name, page number, chunk identifier, and word offsets when available. This metadata is required later for
citation support and error analysis.

# Module 1 — Ingest: extract text from a PDF → page-level text (+ metadata for citations)

## Converted PDF to page-level raw text

In [4]:
def get_paths_pdf_files(folder_path: str) -> List[str]:
    """
    Browse a folder and get path from each PDF files.

    Parameters
    ----------
    folder_path : str
        Path to the folder containing the PDF files.

    Returns
    -------
    results : list of string
    """

    folder = Path(folder_path)

    # return paths all pdf files in folder path
    return list( folder.glob("*.pdf") )

# --- quick sanity check (should not error once implemented) ---
pdf_paths = get_paths_pdf_files(cfg.pdf_path)
rdd = sc.parallelize(pdf_paths)

In [9]:
print(pdf_paths)

[PosixPath('course_material/1-intro.pdf'), PosixPath('course_material/2-spark.pdf'), PosixPath('course_material/3-stream_proc.pdf'), PosixPath('course_material/4-stream_algos.pdf'), PosixPath('course_material/5-nosql.pdf'), PosixPath('course_material/6-parallel.pdf'), PosixPath('course_material/9780521192248_excerpt.pdf'), PosixPath('course_material/A Neural Network Playground.pdf'), PosixPath('course_material/a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf'), PosixPath('course_material/a_ScalableMachineLearningAlgorithmsforBigDataAnalytics-ChallengesandOpportunities.pdf'), PosixPath('course_material/a_the-big-book-of-machine-learning-use-cases.pdf'), PosixPath('course_material/a_TowardScalableSystemsForBigDataAnalytics.pdf'), PosixPath('course_material/Finding Similar Items.pdf'), PosixPath('course_material/Gama_ACMCS_AdaptationCD_accepted (1).pdf'), PosixPath('course_material/Gama_ACMCS_AdaptationCD_accepted.pdf'), PosixPath('course_material/intro.pdf'), PosixP

### Différence entre map et flatMap
La différence fondamentale
map → 1 élément entrant = 1 élément sortant
flatMap → 1 élément entrant = N éléments sortants (la liste résultante est "aplatie")

In [5]:
def extract_pages_from_pdf(pdf_path: str) -> List[Dict]:
    # Read pdf file
    reader = PdfReader(pdf_path)

    # For each page
    pages = list()
    for i in range( len(reader.pages) ):
        # Extract the text
        page = {
            "pdf_name" : Path(pdf_path).name,
            "page_num" : i + 1,
            "text" : reader.pages[i].extract_text() or ""
        }
        # Add the text to the list of pages
        pages.append(page)

    return pages

# flatMap : Regroupe tout les pages de chaque pdf dans une seule liste
pages_rdd = rdd.flatMap(extract_pages_from_pdf)

# 2. **Preprocess / clean** noisy PDF text -  cleaned as needed
Common PDF extraction artifacts such as hyphenation across line breaks, spurious line breaks, repeated whitespace, empty pages, headers/footers when they harm retrieval, and duplicated boilerplate. 
All preprocessing and cleaning procedures must be documented in the report.

In [6]:
def clean_text(page_from_pdf: dict) -> dict:
    """Basic cleanup for PDF-extracted text."""
    text = page_from_pdf["text"]
    # 1. Supprimer les césures en fin de ligne
    text = text.replace("-\n", "")
    
    # 2. Remplacer les retours à la ligne restants par des espaces
    text = text.replace("\n", " ")
    
    # 3. Réduire les espaces multiples à un seul espace
    text = re.sub(r"\s+", " ", text)
    
    # 4. Supprimer les espaces en début et fin
    page_from_pdf["text"] = text.strip()
    
    return page_from_pdf

# map : traitement chaque page
clean_rdd = pages_rdd.map(clean_text)

In [7]:
# filter : supprimer les pages vides
#final_rdd = clean_rdd.filter(lambda x: len(x["text"]) > 50)

In [8]:
print(clean_rdd.count(), "pages extraites")
print(clean_rdd.first())

1662 pages extraites
{'pdf_name': 'pdf_name', 'page_num': 'page_num', 'text': 'text'}


3. **Chunk** the text into fixed-size windows with overlap


5. **Store** a lightweight local index

6. **Retrieve** Top-*k* chunks for a query using similarity search

7. **Construct prompts** that ground the model and request citations

8. **Generate** questions with a free local model

9. **Evaluate** retrieval and generation with a tiny gold set + manual inspection